# Flight delay prediction: end-to-end notebook
Run this from the **project root folder** (`jupyter notebook` there, then open this file) after you have Parquet data in `data/parquet/`
(real BTS data via `python -m src.download_data ...`, or fake data via `python scripts/make_sample_data.py`).

**What you will learn here:** querying data that is too big for RAM (DuckDB), EDA at scale, **data leakage**, time-based splits,
history features, baselines, LightGBM, calibration, and explanations.

In [ ]:
import os, sys
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, matplotlib.pyplot as plt, lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from src import features as F
from src.config import FEATURES, CAT_FEATURES, TARGET
plt.rcParams['figure.figsize'] = (8, 3.5)

## 1. Open the data with DuckDB
DuckDB queries Parquet files directly from disk, so even 100M rows work on a laptop. We only pull **small aggregated results** into pandas.

In [ ]:
con = F.connect()
con.sql("SELECT count(*) AS rows, min(flight_date) AS first_day, max(flight_date) AS last_day FROM flights").df()

In [ ]:
con.sql("""
SELECT Year, count(*) AS flights,
       round(avg(arr_del15), 3) AS late_rate,
       round(avg(cancelled), 3) AS cancel_rate
FROM flights GROUP BY Year ORDER BY Year
""").df()

## 2. Define the target and clean
**Target:** `arr_del15` = 1 if the flight arrived 15+ minutes late. We drop cancelled/diverted flights (they have no arrival delay).
`F.create_base` does this once and stores the result in a temporary table called `base`.

In [ ]:
F.create_base(con)
F.create_stats(con)
con.sql("SELECT count(*) AS usable_flights, round(avg(y),3) AS late_rate FROM base").df()

## 3. Exploratory analysis (aggregate in SQL, plot the small result)

In [ ]:
by_hour = con.sql("SELECT dep_hour, avg(y) AS late_rate FROM base GROUP BY 1 ORDER BY 1").df()
by_hour.plot(x='dep_hour', y='late_rate', kind='bar', legend=False, title='Late rate by scheduled departure hour')
plt.ylabel('share of flights late'); plt.show()

In [ ]:
by_month = con.sql("SELECT month, avg(y) AS late_rate FROM base GROUP BY 1 ORDER BY 1").df()
by_month.plot(x='month', y='late_rate', kind='bar', legend=False, title='Late rate by month'); plt.show()

In [ ]:
con.sql("""
SELECT carrier, count(*) AS flights, round(avg(y),3) AS late_rate
FROM base GROUP BY 1 HAVING count(*) > 1000 ORDER BY late_rate DESC LIMIT 10
""").df()

In [ ]:
con.sql("""
SELECT origin, count(*) AS flights, round(avg(y),3) AS late_rate
FROM base GROUP BY 1 HAVING count(*) > 5000 ORDER BY late_rate DESC LIMIT 10
""").df()

## 4. Data leakage: the most important lesson
A column is **leaky** if you would not know it *before the flight leaves*. The BTS file contains many (`DepDelay`, `TaxiOut`, `WheelsOff`, `ArrTime`, `ActualElapsedTime`, `AirTime`, cause-of-delay columns...).
Watch what happens if we let a model peek at `dep_delay`:

In [ ]:
leak = con.sql("""
    SELECT (crs_dep // 100) % 24 AS dep_hour, month, dow, distance, dep_delay, arr_del15 AS y
    FROM flights
    WHERE cancelled = 0 AND diverted = 0 AND arr_del15 IS NOT NULL AND dep_delay IS NOT NULL
    USING SAMPLE 200000 ROWS
""").df()
n = int(len(leak) * 0.8)
cols_ok = ['dep_hour', 'month', 'dow', 'distance']
for name, cols in [('WITHOUT dep_delay (honest)', cols_ok), ('WITH dep_delay (LEAKY)', cols_ok + ['dep_delay'])]:
    m = lgb.LGBMClassifier(n_estimators=150, verbose=-1).fit(leak.iloc[:n][cols], leak.iloc[:n]['y'])
    print(f"{name:30s} AUC = {roc_auc_score(leak.iloc[n:]['y'], m.predict_proba(leak.iloc[n:][cols])[:,1]):.3f}")

The leaky model looks amazing, but it is useless: when a passenger asks *before* the flight, `dep_delay` does not exist yet.
**Rule:** if your score looks too good to be true, hunt for leakage.

## 5. Time-based split + history features
* **Never split randomly** for forecasting. Train on the past, validate on a later year, test on the latest year.
* **History features** (`route_rate`, `origin_hour_rate`, ...) for a flight in year *Y* come from year *Y-1* only. The model never sees its own label inside a feature.

In [ ]:
years = F.available_years(con)
split = F.split_years(years)
print(split)
train = F.build_dataset(con, split['train'], max_rows=500_000)
val   = F.build_dataset(con, split['val'],   max_rows=200_000)
test  = F.build_dataset(con, split['test'],  max_rows=200_000)
print(len(train), len(val), len(test))
train[FEATURES + [TARGET]].head()

## 6. Baselines (always beat these first)
Accuracy is useless here (~80% of flights are on time). We use **ROC-AUC** and **PR-AUC**.

In [ ]:
cats = F.make_categories(train)
Xtr, ytr = F.apply_categories(train, cats)[FEATURES], train[TARGET]
Xva, yva = F.apply_categories(val, cats)[FEATURES], val[TARGET]
Xte, yte = F.apply_categories(test, cats)[FEATURES], test[TARGET]
base_rate = ytr.mean()

def report(name, y, p):
    print(f"{name:28s} ROC-AUC {roc_auc_score(y,p):.3f}   PR-AUC {average_precision_score(y,p):.3f}")

report('base rate (no skill)', yte, np.full(len(yte), base_rate))
report('last year route rate', yte, Xte['route_rate'].fillna(base_rate))
report('last year origin-hour rate', yte, Xte['origin_hour_rate'].fillna(base_rate))

## 7. Train LightGBM with early stopping on the validation year

In [ ]:
model = lgb.LGBMClassifier(objective='binary', n_estimators=1000, learning_rate=0.05, num_leaves=63,
                           min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           cat_smooth=50, random_state=42, verbose=-1)
model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='auc', categorical_feature=CAT_FEATURES,
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
p_te = model.predict_proba(Xte)[:,1]
report('LightGBM (test year)', yte, p_te)

## 8. Calibration
If the model says 30%, about 30% of those flights should really be late. We check by grouping predictions into bins.

In [ ]:
from sklearn.calibration import calibration_curve
frac, mean_pred = calibration_curve(yte, p_te, n_bins=10, strategy='quantile')
plt.plot([0,max(mean_pred)],[0,max(mean_pred)],'--',color='grey',label='perfect')
plt.plot(mean_pred, frac, marker='o', label='model'); plt.xlabel('predicted'); plt.ylabel('actual'); plt.legend(); plt.show()

## 9. What did the model learn? Feature importance + per-flight explanation

In [ ]:
imp = pd.Series(model.booster_.feature_importance(importance_type='gain'), index=FEATURES).sort_values()
imp.plot(kind='barh', figsize=(7,5), title='Feature importance (gain)'); plt.show()

In [ ]:
# LightGBM can return SHAP-style contributions natively (last column is the bias term)
contrib = model.booster_.predict(Xte.iloc[:1], pred_contrib=True)[0][:-1]
pd.Series(contrib, index=FEATURES).sort_values(key=abs, ascending=False).head(6)

## 10. Use the saved model exactly like the web app does
Run `python -m src.train` first (it saves the full model + lookup tables into `artifacts/`).

In [ ]:
import datetime as dt
from src.predictor import Predictor
pred = Predictor()
o = pred.origins()[0]['code']; d = pred.destinations(o)[0]['code']; c = pred.carriers(o, d)[0]['code']
out = pred.predict(dt.date.today() + dt.timedelta(days=7), '18:30', c, o, d)
print(o, d, c, f"-> {out['probability']:.1%} chance of delay ({out['risk']})")
for r in out['reasons']: print(' ', r['direction'], '|', r['text'])

## Next ideas
Add weather (Meteostat), add the aircraft's previous flight delay (tail-number rotation), tune with Optuna, track runs with MLflow, retrain monthly.